# Метрики сегментации с нуля

Ноутбук к теме [`lesson.md`](lesson.md). Считаем mIoU, Dice и PQ собственным кодом и проверяем утверждения лекции на масках.

1. Пиксельная точность против IoU
2. Dice и IoU: проверяем формулу связи
3. Чем занят mIoU: вклад редких классов
4. PQ = SQ × RQ
5. Усреднять по кадрам или копить по датасету — разные числа

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

C_BG, C_INK, C_GRAY = "#faf9f5", "#141413", "#b0aea5"
C_PANEL, C_ORANGE, C_BLUE, C_GREEN = "#e8e6dc", "#d97757", "#6a9bcc", "#788c5d"
plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG, "axes.edgecolor": C_GRAY,
    "text.color": C_INK, "xtick.color": C_INK, "ytick.color": C_INK,
    "axes.labelcolor": C_INK, "font.size": 11, "legend.frameon": False,
})

def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 1.0

def dice(a, b):
    s = a.sum() + b.sum()
    return float(2 * np.logical_and(a, b).sum() / s) if s else 1.0

## 1. Пиксельная точность против IoU

Сцена с сильным дисбалансом: крупная «дорога», средний «объект» и крошечный «знак». Сравним честную модель с вырожденной, предсказывающей только фон и дорогу.

In [2]:
N = 300
yy, xx = np.mgrid[0:N, 0:N]

road = yy > N * 0.70
obj = (xx - N * 0.35) ** 2 + (yy - N * 0.45) ** 2 < (N * 0.14) ** 2
sign = (xx - N * 0.78) ** 2 + (yy - N * 0.30) ** 2 < (N * 0.035) ** 2

gt = np.zeros((N, N), dtype=int)   # 0 — фон
gt[road] = 1
gt[obj] = 2
gt[sign] = 3
names = {0: 'фон', 1: 'дорога', 2: 'объект', 3: 'знак'}

for k, nm in names.items():
    print(f"{nm:>8}: {(gt == k).mean():6.2%} пикселей")

     фон: 63.81% пикселей
  дорога: 29.67% пикселей
  объект:  6.14% пикселей
    знак:  0.39% пикселей


In [3]:
def pixel_accuracy(pred, gt):
    return float((pred == gt).mean())

def per_class_iou(pred, gt, n_classes):
    """IoU по каждому классу; None для классов, отсутствующих и там, и там."""
    out = []
    for k in range(n_classes):
        p, g = pred == k, gt == k
        if not p.any() and not g.any():
            out.append(None)          # класс не встречается — не определён
        else:
            out.append(iou(p, g))
    return out

def m_iou(pred, gt, n_classes):
    vals = [v for v in per_class_iou(pred, gt, n_classes) if v is not None]
    return float(np.mean(vals))

# вырожденная модель: знает только фон и дорогу
lazy = np.zeros_like(gt)
lazy[road] = 1

# честная модель: всё нашла, но границы слегка раздуты
honest = np.zeros_like(gt)
honest[binary_dilation(road, iterations=2)] = 1
honest[binary_dilation(obj, iterations=3)] = 2
honest[binary_dilation(sign, iterations=2)] = 3

print(f"{'модель':>12} {'pixel acc':>10} {'mIoU':>8}   IoU по классам")
for nm, pred in (("ленивая", lazy), ("честная", honest)):
    pcs = per_class_iou(pred, gt, 4)
    txt = "  ".join(f"{names[i]} {v:.2f}" for i, v in enumerate(pcs) if v is not None)
    print(f"{nm:>12} {pixel_accuracy(pred, gt):>10.2%} {m_iou(pred, gt, 4):>8.3f}   {txt}")

      модель  pixel acc     mIoU   IoU по классам
     ленивая     93.47%    0.477   фон 0.91  дорога 1.00  объект 0.00  знак 0.00
     честная     98.38%    0.893   фон 0.97  дорога 0.98  объект 0.88  знак 0.74


Пиксельная точность у ленивой модели почти не отличается от честной — она полностью определяется двумя крупными классами. mIoU разводит их однозначно, потому что провал на «знаке» входит в среднее с полным весом.

## 2. Dice и IoU: проверяем формулу связи

Утверждение лекции: $\mathrm{Dice} = 2\,\mathrm{IoU}/(1+\mathrm{IoU})$. Проверим не выводом, а замером на масках со случайными искажениями.

In [4]:
rng = np.random.default_rng(0)
errs = []
rows = []
base = (xx - N * 0.4) ** 2 + (yy - N * 0.5) ** 2 < (N * 0.2) ** 2
for _ in range(200):
    dx, dy = rng.integers(-40, 40, 2)
    grow = int(rng.integers(0, 8))
    pred = np.roll(np.roll(base, dy, axis=0), dx, axis=1)
    if grow:
        pred = binary_dilation(pred, iterations=grow)
    i, d = iou(base, pred), dice(base, pred)
    errs.append(abs(d - 2 * i / (1 + i)))
    rows.append((i, d))

print(f"проверено пар масок: {len(rows)}")
print(f"максимальное отклонение от формулы: {max(errs):.2e}")
print(f"Dice >= IoU во всех случаях: {all(d >= i - 1e-12 for i, d in rows)}")

gap = [(i, d - i) for i, d in rows]
worst = max(gap, key=lambda t: t[1])
print(f"наибольший разрыв Dice-IoU: {worst[1]:.3f} при IoU {worst[0]:.2f}")

проверено пар масок: 200
максимальное отклонение от формулы: 1.11e-16
Dice >= IoU во всех случаях: True
наибольший разрыв Dice-IoU: 0.172 при IoU 0.42


## 3. Чем занят mIoU: вклад редких классов

Ключевой вопрос практики: куда выгоднее вложить усилия. Улучшим по очереди каждый класс на одну и ту же величину и посмотрим на прирост mIoU.

In [5]:
def improve(pred, gt, cls, iterations=2):
    """Приблизить предсказание класса cls к истине, ужав раздутую маску."""
    out = pred.copy()
    from scipy.ndimage import binary_erosion
    m = binary_erosion(pred == cls, iterations=iterations)
    out[pred == cls] = 0
    out[m] = cls
    return out

base_miou = m_iou(honest, gt, 4)
print(f"исходный mIoU: {base_miou:.4f}\n")
print(f"{'улучшаем класс':>16} {'его IoU было':>13} {'стало':>7} {'ΔmIoU':>9}")
for cls in (1, 2, 3):
    before = per_class_iou(honest, gt, 4)[cls]
    upd = improve(honest, gt, cls)
    after = per_class_iou(upd, gt, 4)[cls]
    d = m_iou(upd, gt, 4) - base_miou
    print(f"{names[cls]:>16} {before:>13.3f} {after:>7.3f} {d:>+9.4f}")

исходный mIoU: 0.8934

  улучшаем класс  его IoU было   стало     ΔmIoU
          дорога         0.978   0.964   -0.0048
          объект         0.883   0.958   +0.0210
            знак         0.738   1.000   +0.0661


Одна и та же по величине правка границы даёт совершенно разный прирост mIoU. Самый мелкий класс отзывается втрое сильнее среднего и на порядок сильнее крупного: усреднение по классам невзвешенное, а у мелкого объекта та же абсолютная ошибка стоит гораздо большей доли IoU.

У «дороги» прирост вышел отрицательным — сжатие на 2 пикселя её перекорректировало: маска и так была близка к истине, и правка увела её в другую сторону. Это тоже часть вывода: вкладываться в класс, у которого IoU уже 0.98, бессмысленно вдвойне.

## 4. PQ = SQ × RQ

Panoptic Quality по определению, с сопоставлением сегментов по IoU > 0.5.

In [6]:
def panoptic_quality(gt_masks, pred_masks):
    """Возвращает (PQ, SQ, RQ, TP, FP, FN). Порог 0.5 делает сопоставление однозначным."""
    matched, used = [], set()
    for g in gt_masks:
        for pi, p in enumerate(pred_masks):
            if pi in used:
                continue
            v = iou(g, p)
            if v > 0.5:
                matched.append(v)
                used.add(pi)
                break
    tp = len(matched)
    fp, fn = len(pred_masks) - tp, len(gt_masks) - tp
    sq = float(np.mean(matched)) if tp else 0.0
    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) else 0.0
    return sq * rq, sq, rq, tp, fp, fn

def disc(cx, cy, r):
    return (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2

gt_masks = [disc(80, 90, 42), disc(185, 100, 36), disc(125, 220, 34)]

scenarios = {
    "всё найдено, обведено грубо": [disc(84, 94, 36), disc(189, 104, 30), disc(129, 224, 28)],
    "найдено 2 из 3, обведено точно": [disc(81, 91, 41), disc(186, 101, 35)],
    "найдено всё + лишний сегмент": [disc(82, 92, 40), disc(187, 102, 34),
                                     disc(126, 221, 33), disc(240, 40, 25)],
}

print(f"{'сценарий':>32} {'PQ':>7} {'SQ':>7} {'RQ':>7}   TP/FP/FN")
for nm, preds in scenarios.items():
    pq, sq, rq, tp, fp, fn = panoptic_quality(gt_masks, preds)
    print(f"{nm:>32} {pq:>7.3f} {sq:>7.3f} {rq:>7.3f}   {tp}/{fp}/{fn}")

                        сценарий      PQ      SQ      RQ   TP/FP/FN
     всё найдено, обведено грубо   0.702   0.702   1.000   3/0/0
  найдено 2 из 3, обведено точно   0.754   0.943   0.800   2/0/1
    найдено всё + лишний сегмент   0.774   0.903   0.857   3/1/0


Разложение сразу показывает природу проблемы: в первом сценарии страдает SQ (нашли всё, обвели плохо), во втором — RQ (обвели отлично, но пропустили объект). Одно число PQ этого бы не сказало.

И обратите внимание на порядок: сценарий с **пропущенным объектом** получил PQ выше (0.754), чем сценарий, где найдено всё, но обведено грубо (0.702). PQ — это произведение, и провал любого множителя тянет итог вниз одинаково сильно. Если для вашей задачи пропуск объекта дороже неточной обводки, смотреть надо на SQ и RQ по отдельности, а не на их произведение.

## 5. Усреднять по кадрам или копить по датасету

Тонкость, на которой легко получить несравнимые числа. IoU можно считать двумя способами: усреднить по изображениям или накопить TP, FP и FN по всему датасету и посчитать один раз.

In [7]:
rng = np.random.default_rng(3)
per_image, tot_i, tot_u = [], 0, 0
for _ in range(60):
    r = int(rng.integers(4, 60))              # объекты очень разного размера
    cx, cy = rng.integers(80, 220, 2)
    g = disc(cx, cy, r)
    p = binary_dilation(np.roll(g, int(rng.integers(-3, 4)), axis=0), iterations=2)
    per_image.append(iou(g, p))
    tot_i += np.logical_and(g, p).sum()
    tot_u += np.logical_or(g, p).sum()

avg = float(np.mean(per_image))
dataset = float(tot_i / tot_u)
print(f"среднее IoU по кадрам:      {avg:.3f}")
print(f"IoU, накопленный по набору: {dataset:.3f}")
print(f"расхождение:                {abs(avg - dataset):.3f}")
print(f"\nминимальный IoU на кадре: {min(per_image):.3f} (мелкий объект тянет среднее вниз)")

среднее IoU по кадрам:      0.839
IoU, накопленный по набору: 0.912
расхождение:                0.073

минимальный IoU на кадре: 0.464 (мелкий объект тянет среднее вниз)


Числа расходятся, и тем сильнее, чем шире разброс размеров: среднее по кадрам даёт каждому изображению одинаковый вес, а накопление по набору — вес, пропорциональный площади. Оба варианта встречаются в статьях и библиотеках, поэтому при сравнении результатов способ агрегирования нужно уточнять наравне с самой метрикой.